# People Ops Agent — Demo
### Self-serve bonus explainer, payslip generator, and people-ops Q&A

**The problem this solves:** in a lot of companies (including plenty I've seen in Nigeria), employees who can't easily reach the HRIS self-service portal end up messaging the payroll or comp person directly to ask "how was my bonus calculated?" or "can you send me my payslip?" — a slow, repetitive, manual load on a small team.

This agent handles both, plus general people-ops questions, using the transparent bonus engine and synthetic company policy knowledge base built for this project. Try it yourself by editing the employee ID below and re-running.

**Note on bonus components:** the bonus engine supports four possible components — Performance Bonus (everyone), Sales Commission (commission-eligible roles only), Retention Bonus (manager-flagged only), and Spot Bonus (recognition-flagged only). Not every employee qualifies for all four in a given cycle — an Engineering employee with no retention flag and no spot recognition will legitimately only see a Performance Bonus line, while a flagged Sales employee will see all four. This demo deliberately uses an employee who qualifies for all four so you can see the full breakdown in action.


In [1]:
import sys
from pathlib import Path

# This notebook lives in notebook/, but PeopleOpsAgent lives in src/ —
# add src/ to the import path so the plain "from people_ops_agent import ..."
# below works no matter where you launched Jupyter from.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from people_ops_agent import PeopleOpsAgent

agent = PeopleOpsAgent()


### 1. Explaining a bonus in plain English

`EMP1335` is a Sales employee who this cycle: hit part of their sales target, was flagged by their manager as a retention priority, and received a spot recognition — so all four bonus components apply. This is the "full complexity" example; most employees will see a subset of these lines depending on their role and what happened that cycle.

In [2]:
print(agent.handle("Hi, can you explain my bonus? My employee ID is EMP1335"))


Hi James, here's how your bonus was calculated under the FY2026 Bonus Plan v1.0:

• Performance Bonus: USD 4,990.00
   Your performance rating was 3.5/5, which maps to a 5% performance bonus rate under the FY2026 Bonus Plan v1.0 rating table. 5% x USD 99,800 base salary = USD 4,990.00.
• Sales Commission: USD 2,742.01
   You hit 78.5% of your sales target. Commission is 10% of your USD 34,930 commission base for the first 100% of target. Total commission = USD 2,742.01.
• Retention Bonus: USD 7,984.00
   Your manager flagged you as a retention priority this cycle, which triggers a flat 8% of base salary retention bonus: 8% x USD 99,800 = USD 7,984.00.
• Spot Bonus: USD 500.00
   You were recognized for a specific moment of excellence this cycle, which comes with a flat spot bonus of USD 500.00.

Total bonus: USD 16,216.01


### 2. Generating a payslip on demand (no HRIS portal needed)

In [3]:
print(agent.handle("Can I get my payslip? EMP1335"))

# Preview/download the PDF. Colab sandboxes local file previews, so this
# triggers a browser download in Colab; in a local Jupyter/VS Code session
# it shows an inline preview instead.
try:
    from google.colab import files
    files.download(str(REPO_ROOT / "output" / "payslip_EMP1335.pdf"))
    print("Payslip downloaded to your computer — check your Downloads folder.")
except ImportError:
    from IPython.display import IFrame, display
    display(IFrame(str(REPO_ROOT / "output" / "payslip_EMP1335.pdf"), width=600, height=400))


Done - I've generated a payslip for James Martinez (EMP1335) for Current Period: /tmp/project2/output/payslip_EMP1335.pdf


### 3. General people-ops questions (leave, benefits, tax, salary review cycle)

In [4]:
questions = [
    "How much annual leave do I get?",
    "Why is my net pay lower than usual this month?",
    "What's the deal with benefits enrollment?",
    "When is my next salary review?",
]
for q in questions:
    print(f"Employee: {q}")
    print(f"Agent: {agent.handle(q)}\n")


Employee: How much annual leave do I get?
Agent: Employees accrue 20 days of annual leave per year (pro-rated in your first year), plus 10 paid sick days. Leave requests go through the HRIS leave module and need manager approval at least 3 working days in advance for planned leave.

Employee: Why is my net pay lower than usual this month?
Agent: Your payslip breaks down gross pay, statutory deductions (tax, pension), and any benefits contributions to arrive at net pay. If a specific month looks off, the most common causes are a bonus payment moving you into a higher tax bracket for that period, a benefits election change, or a one-off adjustment — I can pull your payslip so we can look at the actual line items together.

Employee: What's the deal with benefits enrollment?
Agent: Benefits enrollment (health insurance, dependents, pension/retirement contributions) opens once a year during Open Enrollment, and also triggers on qualifying life events (marriage, new child, relocation). Reac

### 4. Try your own employee ID

Any ID from `EMP1000` to `EMP2199` exists in the synthetic dataset. Swap the ID below and re-run — most employees will show just a Performance Bonus; commission-eligible Sales employees will also show Sales Commission; a smaller subset will show Retention or Spot Bonus lines too, depending on what was flagged for them that cycle.


In [5]:
print(agent.handle("explain my bonus EMP1050"))


Hi Debra, here's how your bonus was calculated under the FY2026 Bonus Plan v1.0:

• Performance Bonus: USD 1,287.00
   Your performance rating was 3.0/5, which maps to a 3% performance bonus rate under the FY2026 Bonus Plan v1.0 rating table. 3% x USD 42,900 base salary = USD 1,287.00.
• Sales Commission: USD 2,348.35
   You hit 137.6% of your sales target. Commission is 10% of your USD 15,015 commission base for the first 100% of target, plus an accelerated 15% rate on the 37.6% you achieved above target. Total commission = USD 2,348.35.

Total bonus: USD 3,635.35


---
### Where this goes next (production upgrade path)

- Swap the keyword-based `answer_general_question` for a Claude API call (Messages API) with the policy library passed as context — turns this from "smart FAQ" into genuinely open-ended Q&A, with the same agent loop.
- Connect `PeopleOpsAgent` to a real HRIS API instead of a CSV for live data.
- Add authentication so employees can only ask about their own record.
- Deploy as a Slack/Teams bot so employees never leave the tool they already use.
